# Full Finetuning (baseline)

Adapter Layer

LoRA

QLoRA

DoRA (optional / experimental)

MoRA (optional / research)

SBoRA & Multi-SBoRA (research)




In [1]:
%%capture
!pip install -q transformers datasets peft accelerate bitsandbytes trl


In [2]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)

from datasets import Dataset
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)


In [3]:
data = {
    "text": [
        "Deep learning is a subset of machine learning.",
        "Transformers are powerful neural networks.",
        "PEFT methods reduce training cost.",
        "LoRA is widely used for LLM finetuning."
    ]
}

dataset = Dataset.from_dict(data)


In [4]:
MODEL_ID = "facebook/opt-350m"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

tokenized_ds = dataset.map(tokenize, remove_columns=["text"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

# Full Finetuning (Baseline)

In [8]:
model_full = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto"
)

In [9]:
args = TrainingArguments(
    output_dir="./full_ft",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
)

# Add labels to the dataset for Causal Language Modeling
def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

# Create a new dataset with labels
tokenized_ds_with_labels = tokenized_ds.map(add_labels, batched=True)

trainer = Trainer(
    model=model_full,
    args=args,
    train_dataset=tokenized_ds_with_labels, # Use the dataset with labels
)

trainer.train()

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
1,10.948700
2,10.443400


TrainOutput(global_step=2, training_loss=10.696028709411621, metrics={'train_runtime': 0.5863, 'train_samples_per_second': 6.823, 'train_steps_per_second': 3.411, 'total_flos': 931915628544.0, 'train_loss': 10.696028709411621, 'epoch': 1.0})

# Adapter Layer (Classic PEFT)

In [17]:
# !pip install -q adapters
# from adapters import AdapterConfig

# model_adapter = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     device_map="auto"
# )

# model_adapter.add_adapter("my_adapter", AdapterConfig(mh_adapter=True))
# model_adapter.train_adapter("my_adapter")

In [14]:
# args = TrainingArguments(
#     output_dir="./full_ft",
#     per_device_train_batch_size=2,
#     num_train_epochs=1,
#     fp16=True,
#     logging_steps=1,
#     save_strategy="no",
# )

# # Add labels to the dataset for Causal Language Modeling
# def add_labels(examples):
#     examples["labels"] = examples["input_ids"].copy()
#     return examples

# # Create a new dataset with labels
# tokenized_ds_with_labels = tokenized_ds.map(add_labels, batched=True)

# trainer = Trainer(
#     model=model_full,
#     args=args,
#     train_dataset=tokenized_ds_with_labels, # Use the dataset with labels
# )

# trainer.train()

In [ ]:
# trainer = Trainer(
#     model=model_adapter,
#     args=args,
#     train_dataset=tokenized_ds,
# )

# trainer.train()


# LoRA (Low Rank Adaptation)

In [25]:
lora_config = LoraConfig(
    r=4,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
)

model_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)

model_lora = get_peft_model(model_lora, lora_config)
model_lora.print_trainable_parameters()


trainable params: 393,216 || all params: 331,589,632 || trainable%: 0.1186


In [26]:
trainer = Trainer(
    model=model_lora,
    args=args,
    train_dataset=tokenized_ds_with_labels,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
1,11.030700
2,10.262400


TrainOutput(global_step=2, training_loss=10.646575927734375, metrics={'train_runtime': 0.2131, 'train_samples_per_second': 18.767, 'train_steps_per_second': 9.384, 'total_flos': 933123588096.0, 'train_loss': 10.646575927734375, 'epoch': 1.0})

# QLoRA (Quantized LoRA – 4bit)

In [27]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model_qlora = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)


In [28]:
qlora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    task_type=TaskType.CAUSAL_LM,
)

model_qlora = get_peft_model(model_qlora, qlora_config)
model_qlora.print_trainable_parameters()


trainable params: 786,432 || all params: 331,982,848 || trainable%: 0.2369


In [29]:
trainer = Trainer(
    model=model_qlora,
    args=args,
    train_dataset=tokenized_ds_with_labels,
)

trainer.train()

Step,Training Loss
1,10.157400
2,10.116400


TrainOutput(global_step=2, training_loss=10.136889457702637, metrics={'train_runtime': 0.3505, 'train_samples_per_second': 11.413, 'train_steps_per_second': 5.706, 'total_flos': 934331547648.0, 'train_loss': 10.136889457702637, 'epoch': 1.0})

# DoRA (Weight Decomposition – Experimental)

In [30]:
lora_config_dora = LoraConfig(
    r=8,
    lora_alpha=16,
    use_dora=True,   # 🔥 DoRA enabled
    target_modules=["q_proj", "v_proj"],
    task_type=TaskType.CAUSAL_LM,
)

model_dora = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
)

model_dora = get_peft_model(model_dora, lora_config_dora)
model_dora.print_trainable_parameters()


trainable params: 835,584 || all params: 332,032,000 || trainable%: 0.2517


In [31]:
trainer = Trainer(
    model=model_dora,
    args=args,
    train_dataset=tokenized_ds_with_labels,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
1,10.948700
2,10.443400


TrainOutput(global_step=2, training_loss=10.696028709411621, metrics={'train_runtime': 1.1822, 'train_samples_per_second': 3.383, 'train_steps_per_second': 1.692, 'total_flos': 934482542592.0, 'train_loss': 10.696028709411621, 'epoch': 1.0})

# MoRA (Multiplicative Rank Adaptation – Research)

In [32]:
# Conceptual example (research level)
# MoRA = LoRA variant with multiplicative scaling
print("MoRA is currently research-only and not fully stable in PEFT.")


MoRA is currently research-only and not fully stable in PEFT.


# SBoRA & Multi-SBoRA (Structured Block LoRA)

In [33]:
# Conceptual / research implementation
print("SBoRA / Multi-SBoRA are research techniques.")
print("Used in custom LLM training pipelines, not stable in HF PEFT yet.")


SBoRA / Multi-SBoRA are research techniques.
Used in custom LLM training pipelines, not stable in HF PEFT yet.
